<a href="https://colab.research.google.com/github/BCODMO/METS-RCN-TimeSeries/blob/main/METS_CSV_to_JSON_LD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://docs.google.com/spreadsheets/d/1LRW-v2ZNbMuxLsRk-mTQ9cfUQh-70g82GllsYKzvHBc/edit

In [ ]:
!pip install -q pygraphml
!pip install networkx ipysigma

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.7/56.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.8 MB/s eta 0:00:00


In [ ]:
import io
import json
import pandas as pd

import gspread
from gspread_dataframe import get_as_dataframe
from google.auth import default

from io import StringIO

from google.colab import output
output.enable_custom_widget_manager()

from pygraphml import GraphMLParser
from pygraphml import Graph as GraphML
import networkx as nx
from ipysigma import Sigma

In [ ]:
GOOGLE_SHEET = "1LRW-v2ZNbMuxLsRk-mTQ9cfUQh-70g82GllsYKzvHBc"

JSONLD_CONTEXT = {
  '@vocab': 'https://schema.org/',
  'geosparql': 'http://www.opengis.net/ont/geosparql#',
  'sf': 'http://www.opengis.net/ont/sf#'
}

DELIMITER = "|"
BASE_ID = "urn:mets"

IGNORE_FIELDS_STARTS_WITH = '_'
IDENTIFIER_FIELD = '@id'

CSV_FIELD = ['keywords']
ID_FIELDS =  [
   "accountablePerson",
   "address",
   "affiliation",
   "contributor",
   "creator",
   "distribution",
   "funder",
   "funding",
   "isBasedOn",
   "itemOffered",
   "location",
   "offeredBy",
   "offers",
   "organizer",
   "provider",
   "recordedAt",
   "serviceOutput",
   "spatialCoverage",
   "subEvent",
   "superEvent",
]

CUSTOM_FIELDS = {
    'geosparql:asWKT': {
        'func': 'geosparql_as_wkt'
    },
    'keywords': {
        'func': 'parse_csv_row'
    }
}

EVENT_SERIES_TYPE = "time-series"
EVENT_TYPE = "event"
DATASET_TYPE = "dataset"
DATA_DOWNLOAD_TYPE = "data-download"
PLACE_TYPE = "place"
OFFER_TYPE = "offer"
SERVICE_TYPE = "service"
PROPERTYVALUE_TYPE = "property-value"
MONETARY_GRANT_TYPE = "monetary-grant"
PERSON_TYPE = "person"
ORG_TYPE = "organization"
ADDRESS_TYPE = "address"

CONFIG = {
  EVENT_SERIES_TYPE: {
    'gid': '0',
    '@type': 'EventSeries',
    'datatypes': str
  },
  EVENT_TYPE:{
    'gid': '773285762',
    '@type': 'Event',
    'datatypes': str
  },
  DATASET_TYPE: {
    'gid': '949636022',
    '@type': 'Dataset',
    'datatypes': str
  },
  DATA_DOWNLOAD_TYPE: {
    'gid': '1670203551',
    '@type': 'DataDownload',
    'datatypes': str
  },
  PLACE_TYPE: {
    'gid': '835852481',
    '@type': 'Place',
    'datatypes': str
  },
  OFFER_TYPE: {
    'gid': '700306756',
    '@type': 'Offer',
    'datatypes': str
  },
  SERVICE_TYPE: {
    'gid': '1511367588',
    '@type': 'Service',
    'datatypes': str
  },
  PROPERTYVALUE_TYPE: {
    'gid': '801543544',
    '@type': 'PropertyValue',
    'datatypes': str
  },
  MONETARY_GRANT_TYPE: {
    'gid': '613957893',
    '@type': 'MonetaryGrant',
    'datatypes': str
  },
  PERSON_TYPE: {
    'gid': '369091658',
    '@type': 'Person',
    'datatypes': str
  },
  ORG_TYPE: {
    'gid': '303119525',
    '@type': 'Organziation',
    'datatypes': str
  },
  ADDRESS_TYPE: {
    'gid': '478392399',
    '@type': 'PostalAddress',
    'datatypes': str
  }
}

In [ ]:
""" Custom field functions """
def parse_csv_row(field_value, jsonld):
    result = []
    current_field = ""
    in_quotes = False

    for char in field_value:
        if char == '"':
            in_quotes = not in_quotes
        elif char == ',' and not in_quotes:
            result.append(current_field.strip())
            current_field = ""
        else:
            current_field += char

    # Add the last field
    result.append(current_field.strip())

    # Remove surrounding quotes from fields if present
    for i in range(len(result)):
        if result[i].startswith('"') and result[i].endswith('"'):
            result[i] = result[i][1:-1].strip()

    return result

def geosparql_as_wkt(field_value, jsonld):
  # determine the type
  val = field_value.strip().lower()
  geom_type = None
  if val.startswith('point'):
    geom_type = 'sf:Point'
  elif val.startswith('polygon'):
    geom_type = 'sf:Polygon'
  elif val.startswith('multipoint'):
    geom_type = 'sf:MultiPoint'
  elif val.startswith('linestring'):
    geom_type = 'sf:LineString'

  if geom_type is None:
    return field_value

  return {
    'geosparql:hasGeometry': {
      '@type': geom_type,
      'geosparql:asWKT': {
        '@type': "geosparql:wktLiteral",
        '@value': field_value
      },
      'geosparql:crs': {
          '@id': 'http://www.opengis.net/def/crs/OGC/1.3/CRS84'
      }
    }
  }

In [ ]:
def get_sheet_as_dataframe(concept):
  """ Get a Google Sheet tab as Pandas Dataframe """
  gid = CONFIG[concept]['gid']

  sheet_id = "1LRW-v2ZNbMuxLsRk-mTQ9cfUQh-70g82GllsYKzvHBc"
  url = f"https://docs.google.com/spreadsheets/d/{GOOGLE_SHEET}/export?format=csv&gid={gid}"
  return pd.read_csv(url, dtype=CONFIG[concept]['datatypes'])


def get_json_node_id(id_type: str, id: str):
  """ Generate a URI """
  return "{base}:{type}:{id}".format(base=BASE_ID, type=id_type, id=id)


def convert_identifiers(value):
  """ Convert identifiers into URIs """
  if isinstance(value, (list, tuple)):
    for val_idx, val in enumerate(value):
      if isinstance(val, str) and val in ID_GRAPH:
        value[val_idx] = {'@id': ID_GRAPH[val] }
  elif isinstance(value, str) and value in ID_GRAPH:
    value = {'@id': ID_GRAPH[value] }

  return value


def convert_to_jsonld(concept: str):
  """ Convert a Google Sheet tab into JSON-LD """
  df = get_sheet_as_dataframe(concept=concept)
  df = df.to_dict(orient='records')
  for index, ts in enumerate(df):
    cleaned_ts = {k: v for k, v in ts.items() if not pd.isna(v)}
    cleaned_ts = {k: v for k, v in cleaned_ts.items() if not k.startswith(IGNORE_FIELDS_STARTS_WITH)}

    jsonld = {
      '@id': get_json_node_id(id_type=concept, id=ts[IDENTIFIER_FIELD]),
      '@type': CONFIG[concept]['@type']
    }
    ID_GRAPH[ts[IDENTIFIER_FIELD]] = jsonld['@id']
    del cleaned_ts[IDENTIFIER_FIELD]

    jsonld = jsonld | cleaned_ts
    for property in jsonld:
      # determine multi-value fields
      if property in ID_FIELDS and DELIMITER in jsonld[property]:
        jsonld[property] = [item.strip() for item in jsonld[property].split(DELIMITER)]

      # process custom fields
      if property in CUSTOM_FIELDS:
        jsonld[property] = globals()[CUSTOM_FIELDS[property]['func']](jsonld[property], jsonld)

    GRAPH.append(jsonld)

In [ ]:
ID_GRAPH = {} # Keep track of identifiers for creating edges between graph nodes
GRAPH = []    # Build a JSON-LD graph

for key in CONFIG:
  print("Converting {c}".format(c=key))
  convert_to_jsonld(concept=key)

for idx,item in enumerate(GRAPH):
  for property in item:
    if property in ID_FIELDS:
      GRAPH[idx][property] = convert_identifiers(item[property])
  #print(idx, json.dumps(item, indent=4))

Converting time-series
Converting event
Converting dataset
Converting data-download
Converting place
Converting offer
Converting service
Converting property-value
Converting monetary-grant
Converting person
Converting organization
Converting address


In [ ]:
jsonld = {
  '@context': JSONLD_CONTEXT,
  '@graph': GRAPH
}

# Print out your Schema.org JSON-LD
print(json.dumps(jsonld, indent=4))

{
    "@context": {
        "@vocab": "https://schema.org/",
        "geosparql": "http://www.opengis.net/ont/geosparql#",
        "sf": "http://www.opengis.net/ont/sf#"
    },
    "@graph": [
        {
            "@id": "urn:mets:time-series:mets_0001",
            "@type": "EventSeries",
            "name": "Bermuda Atlantic Time-series Study",
            "organizer": [
                {
                    "@id": "urn:mets:person:person_026"
                },
                {
                    "@id": "urn:mets:person:person_027"
                },
                {
                    "@id": "urn:mets:person:person_025"
                },
                {
                    "@id": "urn:mets:person:person_031"
                }
            ],
            "alternateName": "BATS",
            "startDate": "1988-10-10",
            "location": {
                "@id": "urn:mets:place:place_001"
            },
            "funder": {
                "@id": "urn:mets:organization:

In [ ]:
# Generate GraphML for network detection visualization

fname = 'graph.graphml'

def flatten_attributes(attrs):
    flat = {}
    for k, v in attrs.items():
        if k == '@id':
            continue
        if isinstance(v, (str, int, float, bool)):
            flat[k] = v
        elif isinstance(v, dict):
            # Flatten dict with @id if present, or stringify
            flat[k] = v.get('@id', str(v))
        elif isinstance(v, list):
            # Only store list of @ids or join values
            flat[k] = ','.join([i.get('@id', str(i)) for i in v if isinstance(i, dict)])
        else:
            flat[k] = str(v)
    return flat

def jsonld_to_graphml(jsonld_data, output_file=fname):
    G = nx.DiGraph()  # Use DiGraph for directed graph
    for item in jsonld_data:
      node_id = item.get('@id')
      if node_id:
          G.add_node(node_id, **flatten_attributes(item))

      for key, value in item.items():
          if isinstance(value, dict) and '@id' in value:
              G.add_edge(node_id, value['@id'], label=key)
          elif isinstance(value, list):
              for v in value:
                  if isinstance(v, dict) and '@id' in v:
                      G.add_edge(node_id, v['@id'], label=key)
    print(G)
    nx.write_graphml(G, output_file)
    print(f"GraphML written to {output_file}")

jsonld_to_graphml(jsonld_data=jsonld['@graph'], output_file=fname)

DiGraph with 179 nodes and 198 edges
GraphML written to graph.graphml


In [ ]:
# Visualize the graph

g2 = nx.read_graphml(fname)
font_label = 'sans-serif' #'cursive'
Sigma(
    g2,
    node_size=g2.degree,
    default_edge_type='curve',
    node_border_color_from='node',
    node_metrics=['louvain'],
    node_color='louvain',
    node_label='title',
    edge_label='role',
    start_layout=5,
    edge_size=lambda u, v: g2.degree(u) + g2.degree(v),
    edge_size_range=(0.5, 5),
    label_font=font_label,
    node_label_size=g2.degree,
    label_density=0
)

Sigma(nx.DiGraph with 179 nodes and 198 edges)